# PHATPSY Python Driver: Interactive Workflow & 3D Visualization

The **PHATPSY** (Projected Hamiltonian Approach to Polyatomic Systems) package performs non-empirical self-consistent field (SCF) calculations on molecular and chemisorption systems using localized pseudopotentials and projected core/valence basis sets.

This interactive Jupyter Notebook demonstrates:
1. **Dual Input Support**: Loading and inspecting calculations from **either** modern **YAML** or legacy **Namelist / `.stdin`** decks.
2. **Two-Way Conversion**: Converting legacy punch-card formatted decks to structured YAML and vice versa.
3. **End-to-End Execution**: Running calculations through the high-level `PhatpsyDriver` API with automatic format detection.
4. **Cycle-by-Cycle Step-Through**: Stepping through the internal Fortran/C-bridge phases:
   - Common block and state initialization
   - Deck parsing (`&CTLINP`) and title processing
   - Dynamic workspace sizing and array allocation
   - Element basis loading and 0th-order atomic GCHF calculations
   - Polyatomic self-consistent loop cycle-by-cycle convergence
5. **JSON Export & Schema Inspection**: Exporting full quantum observables to JSON.
6. **Data Visualization**:
   - Total energy decomposition and virial ratio check ($-V/T$)
   - Molecular orbital eigenvalue spectrum with HOMO/LUMO identification
   - Unpacking symmetric packed matrices (`overlap_packed` $S_{ij}$) and heatmap plotting
   - Molecular orbital coefficient matrix ($C_{\mu i}$) visualization
7. **Localized Valence-Bond (LVB) Alternating-Spin Theory**:
   - Explicit `spin="up"` / `spin="down"` declarations
   - Automated alternating-spin assignment (Néel / LVB bipartite graph solver)
   - Decoupled electron pairs with zero net molecular spin
8. **Special Atomic Centers & Symmetry Operations**:
   - Virtual / ghost basis centers (`virtual=True`, `QVIRTL=T`) for local coordination neighborhood completion
   - Fixed / frozen atoms (`fixed=True`, `QFIXED=T`)
   - Point-group symmetry operations (`ISYM`, `IXYZ`, `NEQAT`)
9. **Interactive 3D WebGL Visualization**:
   - Embedded Three.js 3D viewer directly inside Jupyter via `deck.show()`
   - Color-coded spin vectors ($\uparrow$ / $\downarrow$), ghost shells, fixed cages, and symmetry planes!

In [ ]:
import sys
import os
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# Locate repository root and configure sys.path
REPO_ROOT = Path("..").resolve() if Path("..").joinpath("phatpsy").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from phatpsy import (
    PhatpsyDriver,
    CalculationDeck,
    ControlConfig,
    AtomConfig,
    SymmetrySpec,
    build_molecule,
    build_diatomic,
    parse_stdin_file,
    parse_stdin_to_deck,
    save_results_to_json,
    unpack_symmetric,
    DIISAccelerator,
    get_element_basis,
    list_available_elements,
    assign_alternating_spin,
    visualize_deck,
    show_deck,
)
import _phatpsy_core as core

print(f"Repository Root: {REPO_ROOT}")
print("Core Fortran Extension (_phatpsy_core): Loaded successfully.")
print(f"Available Basis Elements: {len(list_available_elements())} elements in library.")

## Step 1: Input Format Selection (YAML vs. Legacy Namelist)
PHATPSY supports two input formats:
- **Modern YAML (`.yaml` / `.yml`)**: Clean, human-readable format with explicit spin, virtual, and fixed attributes.
- **Legacy Namelist (`.stdin`)**: Traditional Fortran hybrid deck containing 80-column title cards, `&CTLINP` namelists, and fixed-format punch-card STO tables.

Below, you can select which format to load by setting `input_format = "yaml"` or `"stdin"`. Both are parsed into a unified `CalculationDeck` Python object.

In [ ]:
# Choose input format to load: 'yaml' or 'stdin'
input_format = "yaml"  # Switch between "yaml" and "stdin"

if input_format.lower() == "yaml":
    input_deck_path = REPO_ROOT / "examples" / "n2.yaml"
    deck = CalculationDeck.from_yaml(input_deck_path)
    print(f"Successfully loaded YAML deck from: {input_deck_path}")
else:
    input_deck_path = REPO_ROOT / "examples" / "n2.stdin"
    deck = parse_stdin_to_deck(input_deck_path)
    print(f"Successfully parsed legacy .stdin deck from: {input_deck_path}")

print(f"\nTitle: '{deck.title}'")
print("\n--- Control Configuration ---")
for k, v in deck.control.to_dict().items():
    print(f"  {k:26s} : {v}")

print("\n--- Molecular Geometry ---")
for idx, atom in enumerate(deck.geometry, 1):
    sym_info = f" (Symmetry: {atom.symmetry.to_dict()})" if atom.symmetry else ""
    spin_info = f" [Spin: {atom.spin}]" if atom.spin else ""
    print(f"  Atom {idx}: {atom.element} at {atom.xyz}{spin_info}{sym_info}")

### Format Comparison & Conversion
The unified `CalculationDeck` data model enables seamless two-way conversion:
- `deck.to_yaml()` converts any calculation deck into clean YAML.
- `deck.to_stdin_text()` compiles the YAML into the exact Fortran namelist and fixed-format tables required by the legacy engine.

Let's inspect both representations side-by-side:

In [ ]:
yaml_repr = deck.to_yaml()
stdin_repr = deck.to_stdin_text()

print("=" * 35 + " MODERN YAML FORMAT " + "=" * 35)
print(yaml_repr)

print("=" * 31 + " FORTRAN NAMELIST / DECK FORMAT " + "=" * 31)
print("\n".join(stdin_repr.splitlines()[:28]))
print("... [truncated remaining basis and initial orbital cards] ...")

## Step 2: High-Level Execution via `PhatpsyDriver`
`PhatpsyDriver.run_file()` automatically detects whether the input file is `.yaml`, `.yml`, `.json`, or `.stdin`.

To demonstrate format invariance, we execute the selected input format and then run the alternate format, proving that both produce **identical quantum mechanical energies**.

In [ ]:
driver = PhatpsyDriver()

# Run the user-selected format
print(f"Executing calculation using primary format: {input_deck_path.name}...")
results = driver.run_file(input_deck_path)

# Run the alternate format for validation
alt_path = REPO_ROOT / "examples" / ("n2.stdin" if input_format.lower() == "yaml" else "n2.yaml")
print(f"Executing calculation using alternate format: {alt_path.name}...")
alt_results = driver.run_file(alt_path)

print("\n" + "=" * 60)
print(f"  Primary ({input_deck_path.suffix}):   Total Energy = {results['total_energy']:18.8f} Hartree")
print(f"  Alternate ({alt_path.suffix}): Total Energy = {alt_results['total_energy']:18.8f} Hartree")
diff_energy = abs(results['total_energy'] - alt_results['total_energy'])
print(f"  Energy Difference:     {diff_energy:18.4e} Hartree")
print(f"  SCF Cycles:            {results['cycles']}")
print(f"  Converged:             {results['converged']}")
print("=" * 60)

## Step 3: Stepping Through the Workflow Internals
To understand or customize the calculation, we can step through each phase individually using the bridge modules `core.phatpsy_state_mod` and `core.phatpsy_contrl_mod`.

The phases are:
1. **Phase 3a**: Initialize common block defaults (`state.init_block_data_and_defaults()`)
2. **Phase 3b**: Read title & `&CTLINP` namelist (`contrl.contrl_read_control_input()`)
3. **Phase 3c**: Compute derived dimensions & allocate Fortran workspace arrays (`state.compute_derived_dimensions_and_allocate()`)
4. **Phase 3d**: Read element basis sets and run 0th-order atomic GCHF (`contrl.contrl_init_elements()`)
5. **Phase 3e**: Step cycle-by-cycle through the polyatomic self-consistent field loop
6. **Phase 3f**: Extract converged array properties into Python

In [ ]:
state = core.phatpsy_state_mod
contrl = core.phatpsy_contrl_mod

# Phase 3a: Initialize
state.init_block_data_and_defaults()
print("[Phase 3a] Initialized Fortran common blocks and state defaults.")

# Feed the stdin stream generated from our deck
stdin_stream_text = deck.to_stdin_text()

saved_stdin_fd = os.dup(0)
r_pipe, w_pipe = os.pipe()
try:
    os.write(w_pipe, stdin_stream_text.encode("utf-8"))
    os.close(w_pipe)
    os.dup2(r_pipe, 0)
    
    eof = contrl.contrl_read_control_input()
    print(f"[Phase 3b] CTLINP read complete. EOF: {eof}, QCHECK: {bool(state.qcheck)}")
    print(f"           NATOM={int(state.natom)}, NORBT={int(state.norbt)}, NELEM={int(state.nelem)}")
    
    # Phase 3c: Dynamic Memory Allocation
    state.compute_derived_dimensions_and_allocate()
    contrl.contrl_init_tables()
    print(f"[Phase 3c] Dynamic workspaces allocated (MSTOT={int(state.mstot)}, NORBT={int(state.norbt)}).")
    
    # Phase 3d: Read Element Basis & 0th-order Atomic GCHF
    eof = contrl.contrl_init_elements()
    print(f"[Phase 3d] Element basis initialized and atomic GCHF solved. EOF: {eof}")
    
    # Phase 3e: Stepping Polyatomic SCF Cycles
    print("\n[Phase 3e] Stepping through Polyatomic SCF cycles...")
    print(" Cycle |    Total Energy (H)    |      Kinetic (H)     |     Potential (H)    | Converged")
    print("-------+------------------------+----------------------+----------------------+----------")
    
    converged = False
    while True:
        if not bool(state.qatom):
            converged = contrl.contrl_step_aloop_and_ewmo()
            cyc = int(state.ncycl)
            print(f"  {cyc:4d} | {float(state.tmole):22.8f} | {float(state.totke):20.8f} | {float(state.totpe):20.8f} | {converged}")
            if converged:
                break
        done = contrl.contrl_step_scf()
        if done:
            break
finally:
    os.dup2(saved_stdin_fd, 0)
    os.close(saved_stdin_fd)
    try:
        os.close(r_pipe)
    except OSError:
        pass

print(f"\nStep-through converged at Cycle {int(state.ncycl)} with E = {float(state.tmole):.8f} Hartree.")

## Step 4: JSON Export and Schema Inspection
We save the calculation results to `data/n2_interactive.json` using `save_results_to_json()` and reload it with Python's standard `json` module. This demonstrates how PHATPSY outputs integrate into automated workflows, pipelines, and databases.

In [ ]:
output_json_path = REPO_ROOT / "data" / "n2_interactive.json"
save_results_to_json(results, output_json_path)

print(f"Results saved to: {output_json_path}")
print(f"File size: {output_json_path.stat().st_size:,} bytes")

# Load and inspect JSON structure
with open(output_json_path, "r") as f:
    json_data = json.load(f)

print("\n--- JSON Output Schema & Data Types ---")
for key, val in json_data.items():
    if isinstance(val, list):
        arr = np.array(val)
        print(f"  {key:20s} : Array shape={arr.shape}, dtype={arr.dtype}")
    else:
        print(f"  {key:20s} : {val} (type={type(val).__name__})")

## Step 5: Energy Breakdown & Virial Theorem Analysis
From the JSON results, we analyze the energy components:
- **Kinetic Energy ($T$)**
- **Electronic Potential Energy ($V_{el}$)**
- **Nuclear Repulsion Energy ($V_{nn}$)**
- **Total Potential Energy ($V = V_{el} + V_{nn}$)**
- **Total Molecular Energy ($E = T + V$)**
- **Virial Ratio ($-V / T$)**: According to the molecular virial theorem, at the equilibrium geometry $-V/T \approx 2.0$.

In [ ]:
ke = json_data["kinetic_energy"]
pe = json_data["potential_energy"]
nuc = json_data["nuclear_repulsion"]
tot = json_data["total_energy"]
v_total = pe + nuc
virial_ratio = -v_total / ke

print(f"Kinetic Energy (T):          {ke:16.8f} Hartree")
print(f"Electronic Potential (V_el): {pe:16.8f} Hartree")
print(f"Nuclear Repulsion (V_nn):    {nuc:16.8f} Hartree")
print(f"Total Potential (V_tot):     {v_total:16.8f} Hartree")
print(f"Total Energy (E):            {tot:16.8f} Hartree")
print(f"Virial Ratio (-V_tot / T):   {virial_ratio:16.8f}  (Theoretical ~ 2.0)")

# Energy components bar chart
fig, ax = plt.subplots(figsize=(8.5, 4.8))
components = ['Kinetic ($T$)', 'Elec. Potential ($V_{el}$)', 'Nuclear ($V_{nn}$)', 'Total Potential ($V$)', 'Total Energy ($E$)']
values = [ke, pe, nuc, v_total, tot]
colors = ['#2b5c8f', '#d95f02', '#2ca02c', '#7570b3', '#e7298a']

bars = ax.bar(components, values, color=colors, width=0.55, edgecolor='black', alpha=0.9)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_ylabel("Energy (Hartree)", fontsize=11)
ax.set_title(f"$N_2$ Molecular Energy Decomposition ($E_{{tot}} = {tot:.4f}$ H, $-V/T = {virial_ratio:.3f}$)", fontsize=12, fontweight='bold')
ax.grid(axis='y', linestyle=':', alpha=0.6)

for bar in bars:
    height = bar.get_height()
    offset = 6 if height >= 0 else -16
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, offset),
                textcoords="offset points",
                ha='center', va='bottom' if height < 0 else 'top',
                fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 6: Molecular Orbital Spectrum & Occupations
The JSON output contains the orbital eigenvalues (`eigenvalues`) and fractional or integer occupations (`occupations`) across spin channels.
Below we tabulate the orbitals and plot the molecular orbital energy level diagram with HOMO/LUMO tags.

In [ ]:
eigenvals = np.array(json_data["eigenvalues"])  # Shape: (norbt, ispin)
occupations = np.array(json_data["occupations"]) # Shape: (norbt, ispin)
norbt = json_data["norbt"]
ispin = json_data["ispin"]

print(f"Number of Orbitals (NORBT): {norbt}")
print(f"Spin Channels:              {ispin}")
print("\nOrbital Index | Alpha Energy (H) | Alpha Occ | Beta Energy (H)  | Beta Occ | State")
print("--------------+------------------+-----------+------------------+----------+---------")
for i in range(norbt):
    e_a = eigenvals[i, 0]
    o_a = occupations[i, 0]
    e_b = eigenvals[i, 1] if ispin > 1 else e_a
    o_b = occupations[i, 1] if ispin > 1 else o_a
    state_desc = "Occupied" if o_a > 0.5 else "Virtual"
    print(f"   MO {i+1:2d}      |    {e_a:12.6f}  |   {o_a:5.2f}   |    {e_b:12.6f}  |   {o_b:5.2f}  | {state_desc}")

# Molecular Orbital Diagram
fig, ax = plt.subplots(figsize=(8, 6.5))

alpha_energies = eigenvals[:, 0]
alpha_occs = occupations[:, 0]

occ_energies = [e for e, o in zip(alpha_energies, alpha_occs) if o > 0.5]
virt_energies = [e for e, o in zip(alpha_energies, alpha_occs) if o <= 0.5]
homo = max(occ_energies) if occ_energies else None
lumo = min(virt_energies) if virt_energies else None

for idx, (ene, occ) in enumerate(zip(alpha_energies, alpha_occs)):
    is_occ = occ > 0.5
    color = '#1f77b4' if is_occ else '#d62728'
    ax.hlines(ene, xmin=0.2, xmax=0.7, colors=color, linewidth=2.8)
    label = f"MO {idx+1} ({ene:.3f} H)"
    if ene == homo:
        label += " [HOMO]"
    elif ene == lumo:
        label += " [LUMO]"
    ax.text(0.75, ene, label, va='center', fontsize=9.5, fontweight='bold' if (ene==homo or ene==lumo) else 'normal')

ax.plot([], [], color='#1f77b4', linewidth=2.8, label=f'Occupied (n={len(occ_energies)})')
ax.plot([], [], color='#d62728', linewidth=2.8, label=f'Virtual (n={len(virt_energies)})')

ax.set_xlim(0.0, 1.4)
ax.set_ylim(min(alpha_energies) - 1.5, max(alpha_energies) + 1.5)
ax.set_xticks([])
ax.set_ylabel(r"Orbital Eigenvalue $\epsilon_i$ (Hartree)", fontsize=11)
title = "$N_2$ Molecular Orbital Spectrum"
if homo is not None and lumo is not None:
    gap = lumo - homo
    title += f" (HOMO: {homo:.3f} H, LUMO: {lumo:.3f} H, Gap: {gap:.3f} H)"
ax.set_title(title, fontsize=12, fontweight='bold')
ax.legend(loc='upper left', frameon=True)
ax.grid(axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## Step 7: Overlap Matrix ($S$) Reconstruction
In the Fortran core, symmetric matrices (such as overlap $S$ and Hamiltonian $H$) are stored in packed lower-triangular format of size $N(N+1)/2$ to minimize memory footprint.
Using `phatpsy.unpack_symmetric`, we expand the packed 1D array into the full $N \times N$ symmetric matrix and visualize it as a heatmap.

In [ ]:
packed_overlap = np.array(json_data["overlap_packed"])
s_packed_a = packed_overlap[:, 0]
s_matrix = unpack_symmetric(s_packed_a, norbt)

print(f"Packed array length:     {len(s_packed_a)} (= {norbt}*({norbt}+1)/2)")
print(f"Reconstructed S shape:   {s_matrix.shape}")
print(f"Diagonal values (S_ii):  {np.round(np.diag(s_matrix), 4)}")
print(f"Max off-diagonal:        {np.max(np.abs(s_matrix - np.eye(norbt))):.4f}")

# Overlap Matrix Heatmap
fig, ax = plt.subplots(figsize=(6.5, 5.5))
cax = ax.imshow(s_matrix, cmap='coolwarm', vmin=-0.2, vmax=1.0)
ax.set_title("Basis Overlap Matrix $S_{ij}$", fontsize=12, fontweight='bold')
ax.set_xlabel("Basis Function Index $j$", fontsize=10)
ax.set_ylabel("Basis Function Index $i$", fontsize=10)
ax.set_xticks(range(norbt))
ax.set_yticks(range(norbt))
ax.set_xticklabels([rf"$\chi_{{{k+1}}}$" for k in range(norbt)])
ax.set_yticklabels([rf"$\chi_{{{k+1}}}$" for k in range(norbt)])

for i in range(norbt):
    for j in range(norbt):
        val = s_matrix[i, j]
        text_color = "white" if abs(val) > 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color, fontsize=8)

fig.colorbar(cax, ax=ax, label="Overlap Integral $S_{ij}$")
plt.tight_layout()
plt.show()

## Step 8: Molecular Orbital Coefficients Matrix ($C$)
The molecular orbital coefficients $C_{\mu i}$ define each canonical molecular orbital $\psi_i$ as a linear combination of atomic basis functions $\chi_\mu$:
$$\psi_i = \sum_{\mu} C_{\mu i} \chi_\mu$$
The coefficient matrix has dimensions $(M_{stot} \times N_{orbt})$. Below we visualize the Alpha orbital coefficient matrix.

In [ ]:
mo_coeffs = np.array(json_data["mo_coefficients"])
c_alpha = mo_coeffs[:, :, 0]

print(f"MO Coefficient Matrix shape: {c_alpha.shape} (MSTOT={c_alpha.shape[0]} x NORBT={c_alpha.shape[1]})")

fig, ax = plt.subplots(figsize=(8, 6.5))
vmax = np.max(np.abs(c_alpha))
cax = ax.imshow(c_alpha, cmap='bwr', vmin=-vmax, vmax=vmax)
ax.set_title(r"Molecular Orbital Coefficients $C_{\mu i}$", fontsize=12, fontweight='bold')
ax.set_xlabel("Molecular Orbital Index $i$", fontsize=10)
ax.set_ylabel(r"Basis Function Index $\mu$", fontsize=10)
ax.set_xticks(range(norbt))
ax.set_yticks(range(c_alpha.shape[0]))
ax.set_xticklabels([f"MO {k+1}" for k in range(norbt)])
ax.set_yticklabels([rf"$\chi_{{{k+1}}}$" for k in range(c_alpha.shape[0])])

fig.colorbar(cax, ax=ax, label=r"Coefficient $C_{\mu i}$")
plt.tight_layout()
plt.show()

## Step 9: Alternating-Spin Localized Valence-Bond (LVB) Representation

A hallmark of PHATPSY is its use of **spin-flipping (`QFLIP=T`)** to create an **alternating-spin Perfect-Pairing Valence-Bond (PP-VB / LVB)** representation—even for nominally "closed-shell" molecules!

### Why Localized Valence-Bond?
* In standard **Restricted Hartree-Fock (RHF)**, pairs of electrons are forced into identical spatial orbitals ($\phi_i(1)\phi_i(2)$), which overestimates unphysical ionic configurations ($A^- \dots B^+$) and fails to dissociate correctly.
* In PHATPSY's **Localized Valence-Bond** approach, electron pairs are decoupled into localized atomic channels:
  - Center 1 is $\alpha$-dominant (`spin="up"`, $+S_z$)
  - Center 2 is $\beta$-dominant (`spin="down"`, $-S_z$)
  - The total molecular spin is strictly zero ($S_{z,\text{tot}} = 0.0000$), but the local pairing directly captures intra-bond electron correlation!

### Automated Alternating-Spin Assignment
The `phatpsy.spin` module analyzes molecular connectivity graphs and automatically assigns alternating `up` and `down` states. Let's demonstrate this on an ethylene ($C_2H_4$) molecule:

In [ ]:
# Build ethylene with automated alternating-spin assignment
xyz_c2h4 = '''6
Ethylene molecule
C  0.000000  0.000000  0.669500
C  0.000000  0.000000 -0.669500
H  0.000000  0.928900  1.232100
H  0.000000 -0.928900  1.232100
H  0.000000  0.928900 -1.232100
H  0.000000 -0.928900 -1.232100
'''

c2h4_deck = build_molecule(xyz_c2h4, basis="SZ", auto_spin_flip=True)

print("--- Automated Alternating-Spin Assignment for Ethylene ---")
for idx, a in enumerate(c2h4_deck.geometry, 1):
    direction = "Alpha-dominant (UP)" if a.spin == "up" else "Beta-dominant (DOWN)"
    print(f"  Atom {idx:2d} ({a.element:<2s}): {direction:24s} at z={a.xyz[2]:8.4f} Bohr")

print("\nCompiled &ATOM cards showing automatic QFLIP=T emission:")
for line in c2h4_deck.to_stdin_text().splitlines():
    if "&ATOM" in line:
        print(" ", line)

## Step 10: Special Centers (Virtual Atoms, Fixed Atoms, and Symmetry)

PHATPSY natively supports three powerful atomic configuration flags:
1. **Virtual (Ghost) Atoms (`virtual=True`, `QVIRTL=T`)**: Centers that retain basis functions and external potentials but carry **no occupied electrons**. Used to complete the local coordination neighborhood and external potentials during local atomic calculations (excluded at the molecular EWMO level).
2. **Fixed (Frozen) Atoms (`fixed=True`, `QFIXED=T`)**: Atoms whose orbitals and local model potentials are frozen throughout SCF iterations. Used for QM/MM-style solvent scaffolds or frozen cores.
3. **Point-Group Symmetry (`SymmetrySpec`)**:
   - `ISYM = -1`: Reflection ($\sigma$) across the plane perpendicular to `IXYZ`
   - `ISYM = -2`: Inversion ($i$) through the origin
   - `ISYM = N`: $N$-fold rotation ($C_n$) about axis `IXYZ`

Below, we construct a custom system showcasing all three features:

In [ ]:
# Nitrogen dimer with a fixed center, a virtual ghost probe, and symmetry reflection
special_atoms = [
    # Primary active Nitrogen atom
    AtomConfig(element="N", xyz=[0.0, 0.0, 1.034], spin="up"),
    
    # Symmetry-equivalent Nitrogen atom (reflected across xy-plane, spin-flipped)
    AtomConfig(
        element="N",
        symmetry=SymmetrySpec(ixyz=1, isym=-1, qflip=True),
        spin="down"
    ),
    
    # Virtual ghost basis center (e.g. for local coordination neighborhood completion)
    AtomConfig(element="H", xyz=[0.0, 2.0, 0.0], virtual=True),
    
    # Fixed / frozen environment atom
    AtomConfig(element="C", xyz=[0.0, -3.0, 0.0], fixed=True, spin="up"),
]

special_deck = CalculationDeck(
    title="Special Features Demonstration Deck",
    geometry=special_atoms,
    control=ControlConfig(spin_unrestricted=True),
)

print("--- Generated Special Features Input Deck ---")
print(special_deck.to_yaml())

## Step 11: Interactive 3D WebGL Visualization in Jupyter

PHATPSY includes an embedded **3D Three.js WebGL visualizer** accessible directly from Python via `deck.show()`.

### Visual Color-Coding:
- **Spin Up ($\alpha$)**: Solid sphere with an upward cyan vector arrow ($\uparrow$)
- **Spin Down ($\beta$)**: Solid sphere with a downward orange vector arrow ($\downarrow$)
- **Virtual (Ghost) Centers**: Translucent wireframe spheres
- **Fixed (Frozen) Atoms**: Metallic gray wireframe bounding cages
- **Symmetry Elements**: Translucent reflection mirror planes (`ISYM=-1`), rotation axes (`ISYM=N`), and dashed links to generator atoms

Rotate, pan, and zoom the interactive 3D model below!

In [ ]:
# Render interactive 3D WebGL widget inline in Jupyter
# You can rotate (left click), pan (right click), zoom (scroll), and toggle features!
special_deck.show()

In [ ]:
# View Ethylene with Localized Valence-Bond alternating spin arrows
c2h4_deck.show()

### Standalone HTML Export & CLI Viewer
You can also export any calculation to a self-contained HTML file or launch the 3D viewer from the command line:

```python
# Export standalone HTML
special_deck.visualize(output_path="special_deck_3d.html", auto_open=False)
```

Or from your terminal:
```bash
# View any calculation deck or legacy stdin in 3D:
phatpsy view examples/ag13.stdin
phatpsy view examples/CO_ni_100.stdin

# Build an XYZ file with automatic alternating spin and view:
phatpsy view molecule.xyz --auto-spin
```

## Summary
The modernized PHATPSY Python driver unites high-performance Fortran physics with modern scientific computing:
- **Localized Valence-Bond (LVB)** alternating-spin theory natively supported and automated.
- Full support for **virtual**, **fixed**, and **point-group symmetry** centers.
- Direct **3D WebGL visualization** in Jupyter and CLI.
- Unified data models bridging YAML, JSON, and legacy Fortran namelists.